In [1]:
!pip install tensorflow nltk pandas scikit-learn


In [2]:
import pandas as pd
import numpy as np

import re 
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense,LSTM,Embedding,Bidirectional,Dropout

In [15]:
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chavd\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
df = pd.read_csv('jigsaw-toxic-comment-train.csv')

LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

df = df[['comment_text'] + LABELS]
df.head()


,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [17]:
x = df['comment_text']
y = df[LABELS].values   # Now y has 6 labels instead of 1


In [18]:
x.isna().sum()

np.int64(0)

In [19]:
len(df)

223549

In [20]:
print(stopwords.words('english')[:20])


['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


In [21]:
stop_words = stopwords.words('english')

In [22]:
corpus = []

for text in df['comment_text']:
    text = str(text)
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [word for word in words if word not in stop_words]
    corpus.append(' '.join(words))

print(corpus[:5])


['explanation edits made username hardcore metallica fan reverted vandalisms closure gas voted new york dolls fac please remove template talk page since retired', 'aww matches background colour seemingly stuck thanks talk january utc', 'hey man really trying edit war guy constantly removing relevant information talking edits instead talk page seems care formatting actual info', 'make real suggestions improvement wondered section statistics later subsection types accidents think references may need tidying exact format ie date format etc later one else first preferences formatting style references want please let know appears backlog articles review guess may delay reviewer turns listed relevant form eg wikipedia good article nominations transport', 'sir hero chance remember page']


In [23]:
tokenizer = Tokenizer(num_words=50000)
tokenizer.fit_on_texts(corpus)

sequences = tokenizer.texts_to_sequences(corpus)


In [24]:
max_len = 150
x = pad_sequences(sequences, maxlen=max_len)

y = df[LABELS].values   # ✅ MULTI-LABEL target (6 outputs)


In [25]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense

model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=50000, output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(6, activation='sigmoid')   # ✅ IMPORTANT: 6 outputs
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 150, 128)       │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,499,590 (24.79 MB)

 Trainable params: 6,499,590 (24.79 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history = model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(x_test, y_test)
)


Epoch 1/3
1398/1398 ━━━━━━━━━━━━━━━━━━━━ 763s 542ms/step - accuracy: 0.8898 - loss: 0.0760 - val_accuracy: 0.9951 - val_loss: 0.0548
Epoch 2/3
1398/1398 ━━━━━━━━━━━━━━━━━━━━ 1003s 718ms/step - accuracy: 0.9700 - loss: 0.0526 - val_accuracy: 0.9950 - val_loss: 0.0540
Epoch 3/3
1398/1398 ━━━━━━━━━━━━━━━━━━━━ 817s 584ms/step - accuracy: 0.9603 - loss: 0.0450 - val_accuracy: 0.9945 - val_loss: 0.0543


In [29]:
def predict_toxic(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower().split()
    text = [word for word in text if word not in stop_words]
    text = ' '.join(text)

    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)

    preds = model.predict(pad)[0]

    results = {}
    for i, label in enumerate(LABELS):
        results[label] = "YES" if preds[i] > 0.5 else "NO"

    return results


In [30]:
print(predict_toxic('you are smart'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 896ms/step
{'toxic': 'NO', 'severe_toxic': 'NO', 'obscene': 'NO', 'threat': 'NO', 'insult': 'NO', 'identity_hate': 'NO'}


In [31]:
print(predict_toxic('you are asshole' ))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
{'toxic': 'YES', 'severe_toxic': 'NO', 'obscene': 'YES', 'threat': 'NO', 'insult': 'YES', 'identity_hate': 'NO'}


In [35]:
print(predict_toxic('...your sentence here...'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
{'toxic': 'NO', 'severe_toxic': 'NO', 'obscene': 'NO', 'threat': 'NO', 'insult': 'NO', 'identity_hate': 'NO'}


In [41]:
print(predict_toxic("People like you should not be allowed here."))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
{'toxic': 'NO', 'severe_toxic': 'NO', 'obscene': 'NO', 'threat': 'NO', 'insult': 'NO', 'identity_hate': 'NO'}
